# Superhedging experiment (local version)

Run the cells top to bottom. `fast_solvers.py` must sit in the same folder as this notebook. Results are written to `awdro_results/time_grid_<id>/` next to this notebook.
Finished seeds are checkpointed, so if the run is interrupted you can re-run from the setup cell and completed seeds are skipped.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%cd /content
!git clone https://github.com/jcvetinovska/awdro.git awdro

/content
fatal: destination path 'awdro' already exists and is not an empty directory.


In [4]:
from pathlib import Path
import sys
from IPython.display import display   # used in cells 8 and 9

root = Path("/content/awdro")

for folder in [
    root,
    root / "hedging",
    root / "path simulation",
    root / "inner maximization",
    root / "outer minimization",
]:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

In [5]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/jcvetinovska/awdro.git"
CHECKOUT = Path("/content/awdro_repo")

if not CHECKOUT.exists():
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(CHECKOUT)],
        check=True,
    )

# Pull deliberately when you want to update:
# subprocess.run(
#     ["git", "-C", str(CHECKOUT), "pull", "--ff-only"],
#     check=True,
# )
# Restart the kernel after updating previously imported code.

required_folders = [
    "hedging",
    "path simulation",
    "inner maximization",
    "outer minimization",
]

# Supports either repository root/hedging or repository root/awdro/hedging.
root = next(
    (
        candidate
        for candidate in [CHECKOUT, CHECKOUT / "awdro"]
        if all((candidate / name).is_dir() for name in required_folders)
    ),
    None,
)

if root is None:
    raise FileNotFoundError(
        "Cannot locate the project folders. "
        "Set root to their parent directory inside the cloned repository."
    )

for folder in [root] + [root / name for name in required_folders]:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

CODE_VERSION = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Project:", root)
print("Commit:", CODE_VERSION)


Project: /content/awdro_repo
Commit: 5c20a371ae6538386f3cfbb03094ea250618748b


In [6]:
from pathlib import Path
import hashlib
import json
import platform
import shutil
import time
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
import cvxpy as cp
import matplotlib.pyplot as plt
from IPython.display import display

from Heston import Heston
from SuperhedgingProblem import SuperhedgingProblem
from NonrectangularSolver import NonrectangularSolver
from RectangularSolver import RectangularSolver
from RelaxedSuperhedgingSolver import RelaxedSuperhedgingSolver
from fast_solvers import FastRectangularSolver, FastRelaxedSuperhedgingSolver

if "HIGHS" not in cp.installed_solvers():
    raise RuntimeError("Restart the kernel after installing highspy.")

# Pilot run. Increase to 50 or 100 after measuring runtime.
N_SEEDS = 20
SEEDS = list(range(1000, 1000 + N_SEEDS))

# 2-4 is the practical range for the exact LPs. 5 works only for the
# nonrectangular solver (about 8 min per seed, ~3 GB RAM); the rectangular LP
# is refused by the size guard in fast_solvers.py. 6+ is infeasible.
N_STEPS_LIST = [2, 3, 4]
BRANCHES = 2

MATURITY = 1.0
BUDGET = 1.0
PENALTY = 10.0
STRIKE = 100.0

CAPITAL_BOUNDS = (0.0, 50.0)
HOLDING_BOUNDS = (-2.0, 2.0)

ABS_TOL = 1e-5
REL_TOL = 1e-5

MODEL_PARAMETERS = {
    "s0": 100.0,
    "v0": 0.04,
    "mu": 0.0,
    "kappa": 2.0,
    "theta": 0.04,
    "vol_of_vol": 0.3,
    "rho": -0.7,
}

# Increment if you change the notebook's experimental logic.
EXPERIMENT_VERSION = "1"

config = {
    "experiment_version": EXPERIMENT_VERSION,
    "git_commit": CODE_VERSION,
    "model": MODEL_PARAMETERS,
    "maturity": MATURITY,
    "budget": BUDGET,
    "penalty": PENALTY,
    "strike": STRIKE,
    "branches": BRANCHES,
    "capital_bounds": CAPITAL_BOUNDS,
    "holding_bounds": HOLDING_BOUNDS,
    "payoff": "European call",
    "cost": "unweighted sum of squared stock and variance differences",
    "abs_tol": ABS_TOL,
    "rel_tol": REL_TOL,
    "python": platform.python_version(),
    "packages": {
        package: version(package)
        for package in [
            "numpy", "pandas", "scipy", "torch",
            "cvxpy", "highspy", "matplotlib",
        ]
    },
}

# Seeds and time grids are excluded so completed runs can be reused.
study_id = hashlib.sha256(
    json.dumps(config, sort_keys=True).encode()
).hexdigest()[:12]

results_root = RESULTS_ROOT
output_dir = results_root / f"time_grid_{study_id}"
output_dir.mkdir(parents=True, exist_ok=True)


def save_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "w") as file:
        json.dump(value, file, indent=2)
    temporary.replace(path)


save_json(output_dir / "configuration.json", config)
save_json(
    output_dir / "requested_runs.json",
    {"seeds": SEEDS, "n_steps": N_STEPS_LIST},
)

print("Results folder:", output_dir.resolve())
print("Paired experiments:", len(SEEDS) * len(N_STEPS_LIST))

ModuleNotFoundError: No module named 'fast_solvers'

In [ ]:
def to_array(x):
    """Convert to float array; fall back to object array if ragged."""
    try:
        return np.asarray(x, dtype=float)
    except ValueError:
        return np.asarray(x, dtype=object)   # load with allow_pickle=True


def call_payoff(paths):
    return torch.clamp(paths[:, -1, 0] - STRIKE, min=0.0)


def run_pair(seed, n_steps):
    started = time.perf_counter()

    row = {
        "seed": int(seed),
        "n_steps": int(n_steps),
        "status": "failed",
        "stage": "construction",
        "error": "",
    }

    try:
        model = Heston(**MODEL_PARAMETERS)
        times = np.linspace(0.0, MATURITY, n_steps + 1)

        problem = SuperhedgingProblem(
            payoff=call_payoff,
            trading_times=times,
        )

        discretization = {
            "seed": int(seed),
            "reference_branches": BRANCHES,
        }

        rectangular = FastRectangularSolver(
            model, times, BUDGET, discretization
        )
        nonrectangular = NonrectangularSolver(
            model, times, BUDGET, discretization
        )

        row["stage"] = "shared_model_check"

        np.testing.assert_array_equal(
            np.asarray([
                node["state"] for node in rectangular.reference_nodes
            ]),
            np.asarray([
                node["state"] for node in nonrectangular.reference_nodes
            ]),
        )

        for t in range(n_steps + 1):
            np.testing.assert_array_equal(
                rectangular.adversarial_candidates[t],
                nonrectangular.adversarial_candidates[t],
            )

        outer = FastRelaxedSuperhedgingSolver(
            problem=problem,
            inner_solver=nonrectangular,
            penalty=PENALTY,
        )

        methods = {
            "nonrectangular": outer.solve_adaptive_hedge_nonrectangular,
            "rectangular": outer.solve_adaptive_hedge_rectangular,
        }

        for name, method in methods.items():
            row["stage"] = f"{name}_optimization"
            solve_started = time.perf_counter()

            result = method(
                rectangular, CAPITAL_BOUNDS, HOLDING_BOUNDS
            )

            row[f"{name}_seconds"] = (
                time.perf_counter() - solve_started
            )

            value = float(result["value"])
            capital = float(result["capital"])
            shortfall = float(result["worst_case_shortfall"])
            holdings = np.asarray(result["holdings"], dtype=float)

            if not np.isfinite([value, capital, shortfall]).all():
                raise RuntimeError(f"{name}: nonfinite scalar result.")

            if holdings.size == 0 or not np.isfinite(holdings).all():
                raise RuntimeError(f"{name}: invalid holdings.")

            np.testing.assert_allclose(
                value,
                capital + PENALTY * shortfall,
                atol=ABS_TOL,
                rtol=REL_TOL,
            )

            capital_tol = ABS_TOL + REL_TOL * max(
                1.0, *map(abs, CAPITAL_BOUNDS)
            )
            holding_tol = ABS_TOL + REL_TOL * max(
                1.0, *map(abs, HOLDING_BOUNDS)
            )
            value_tol = ABS_TOL + REL_TOL * max(1.0, abs(value))

            if shortfall < -value_tol:
                raise RuntimeError(f"{name}: negative shortfall.")

            if not (
                CAPITAL_BOUNDS[0] - capital_tol
                <= capital
                <= CAPITAL_BOUNDS[1] + capital_tol
            ):
                raise RuntimeError(f"{name}: capital bounds violated.")

            if (
                (holdings < HOLDING_BOUNDS[0] - holding_tol).any()
                or (holdings > HOLDING_BOUNDS[1] + holding_tol).any()
            ):
                raise RuntimeError(f"{name}: holding bounds violated.")

            row[f"{name}_value"] = value
            row[f"{name}_capital"] = capital
            row[f"{name}_shortfall"] = shortfall

            row[f"{name}_capital_at_bound"] = bool(
                np.isclose(
                    capital, CAPITAL_BOUNDS,
                    atol=capital_tol, rtol=0.0,
                ).any()
            )

            row[f"{name}_holding_bound_fraction"] = float(
                (
                    np.isclose(
                        holdings, HOLDING_BOUNDS[0],
                        atol=holding_tol, rtol=0.0,
                    )
                    | np.isclose(
                        holdings, HOLDING_BOUNDS[1],
                        atol=holding_tol, rtol=0.0,
                    )
                ).mean()
            )

            arrays = {
                "capital": np.asarray(capital),
                "holdings": holdings,
                "offsets": np.asarray(result["offsets"]),
                "trading_times": times,
            }
            arrays.update({
                f"histories_t{t}": to_array(histories)
                for t, histories in enumerate(result["histories_by_time"])
            })

            np.savez_compressed(
                output_dir / f"T{n_steps}_seed{seed}_{name}_hedge.npz",
                **arrays,
            )

        row["stage"] = "gap_check"

        gap = row["rectangular_value"] - row["nonrectangular_value"]
        tolerance = ABS_TOL + REL_TOL * max(
            1.0,
            abs(row["rectangular_value"]),
            abs(row["nonrectangular_value"]),
        )

        row["gap"] = gap
        row["gap_tolerance"] = tolerance
        row["positive_above_tolerance"] = bool(gap > tolerance)

        if gap < -tolerance:
            raise RuntimeError(
                f"Gap {gap:.6g} is below -tolerance {-tolerance:.6g}."
            )

        if BUDGET == 0 and abs(gap) > tolerance:
            raise RuntimeError("Zero-budget objectives disagree.")

        row["status"] = "ok"
        row["stage"] = "complete"

    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"

    row["total_seconds"] = time.perf_counter() - started
    return row

In [ ]:
records = []

for n_steps in N_STEPS_LIST:
    for seed in SEEDS:
        checkpoint = output_dir / f"T{n_steps}_seed{seed}.json"
        row = None

        if checkpoint.exists():
            with open(checkpoint) as file:
                cached = json.load(file)

            if cached["status"] == "ok":
                row = cached

        if row is None:
            print(
                f"Running {n_steps} intervals, seed {seed}...",
                flush=True,
            )
            row = run_pair(seed, n_steps)
            save_json(checkpoint, row)

        records.append(row)

        # Save the combined table after every experiment.
        pd.DataFrame(records).to_csv(
            output_dir / "results.csv",
            index=False,
        )

        print(
            f"  {row['status']} | "
            f"{row['total_seconds']:.1f} seconds"
        )

        if row["status"] != "ok":
            print(f"  {row['stage']}: {row['error']}")

results = pd.DataFrame(records)

display(
    results.groupby(["n_steps", "status"])
    .size()
    .unstack(fill_value=0)
)

Running 2 intervals, seed 1000...
  ok | 0.0 seconds
Running 2 intervals, seed 1001...
  ok | 0.0 seconds
Running 2 intervals, seed 1002...
  ok | 0.0 seconds
Running 2 intervals, seed 1003...
  ok | 0.0 seconds
Running 2 intervals, seed 1004...
  ok | 0.0 seconds
Running 2 intervals, seed 1005...
  ok | 0.0 seconds
Running 2 intervals, seed 1006...
  ok | 0.0 seconds
Running 2 intervals, seed 1007...
  ok | 0.0 seconds
Running 2 intervals, seed 1008...
  ok | 0.0 seconds
Running 2 intervals, seed 1009...
  ok | 0.0 seconds
Running 2 intervals, seed 1010...
  ok | 0.0 seconds
Running 2 intervals, seed 1011...
  ok | 0.0 seconds
Running 2 intervals, seed 1012...
  ok | 0.0 seconds
Running 2 intervals, seed 1013...
  ok | 0.0 seconds
Running 2 intervals, seed 1014...
  ok | 0.0 seconds
Running 2 intervals, seed 1015...
  ok | 0.0 seconds
Running 2 intervals, seed 1016...
  ok | 0.0 seconds
Running 2 intervals, seed 1017...
  ok | 0.0 seconds
Running 2 intervals, seed 1018...
  ok | 0.0 s

KeyboardInterrupt: 

In [ ]:
if results.empty or "status" not in results:
    raise RuntimeError(
        "No results were produced. Check the errors printed above."
    )

summary_rows = []

for n_steps, group in results.groupby("n_steps"):
    good_group = group.loc[group["status"] == "ok"]

    item = {
        "n_steps": int(n_steps),
        "attempted": len(group),
        "successful": len(good_group),
        "failed": len(group) - len(good_group),
    }

    if len(good_group) >= 2:
        gaps = good_group["gap"].to_numpy(dtype=float)
        rng = np.random.default_rng(2026 + int(n_steps))

        indices = rng.integers(
            0, len(gaps), size=(10_000, len(gaps))
        )
        bootstrap_means = gaps[indices].mean(axis=1)
        low, high = np.quantile(bootstrap_means, [0.025, 0.975])

        item.update({
            "mean_gap": gaps.mean(),
            "median_gap": np.median(gaps),
            "ci_low": low,
            "ci_high": high,
            "positive_fraction": (
                good_group["positive_above_tolerance"].mean()
            ),
            "mean_nonrectangular_value": (
                good_group["nonrectangular_value"].mean()
            ),
            "mean_rectangular_value": (
                good_group["rectangular_value"].mean()
            ),
            "median_seconds": good_group["total_seconds"].median(),
        })

    summary_rows.append(item)

summary = pd.DataFrame(summary_rows)
summary.to_csv(output_dir / "summary.csv", index=False)
display(summary)

if (summary["failed"] > 0).any():
    print(
        "Warning: summaries and plots use successful runs only. "
        "Investigate and report failures."
    )

if N_SEEDS < 30:
    print("This is a runtime pilot, not the final presentation sample.")

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 12,
    "text.color": "#192332",
    "axes.labelcolor": "#394455",
    "xtick.color": "#657083",
    "ytick.color": "#657083",
    "axes.edgecolor": "#CBD1D9",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.dpi": 300,
})

COLORS = ["#2563EB", "#0D9488", "#9333EA", "#EA580C"]

good = results.loc[results["status"] == "ok"].copy()

if "mean_gap" not in summary:
    raise RuntimeError("At least two successful runs per grid are needed.")

plot_summary = summary.dropna(subset=["mean_gap"]).sort_values("n_steps")

subtitle = (
    f"European call · adaptive hedge · ε = {BUDGET:g} · "
    f"κ = {PENALTY:g} · maturity = {MATURITY:g}"
)

failure_note = (
    " Successful runs only; failures recorded."
    if (summary["failed"] > 0).any() else ""
)


def save_figure(fig, filename):
    fig.text(
        0.01, 0.01,
        "Finite-model study · unweighted squared path cost." + failure_note,
        fontsize=9,
        color="#778193",
    )
    fig.tight_layout(rect=[0, 0.06, 1, 0.92])
    fig.savefig(output_dir / f"{filename}.pdf", bbox_inches="tight")
    fig.savefig(output_dir / f"{filename}.png", bbox_inches="tight")
    plt.show()

In [ ]:
# Figure 1: raw gaps, means and uncertainty intervals.

fig, ax = plt.subplots(figsize=(9, 5.2))

for index, (_, stats) in enumerate(plot_summary.iterrows()):
    steps = int(stats["n_steps"])
    color = COLORS[index % len(COLORS)]

    gaps = good.loc[good["n_steps"] == steps, "gap"].to_numpy()
    jitter = np.random.default_rng(steps).uniform(-0.12, 0.12, len(gaps))

    ax.scatter(
        steps + jitter, gaps,
        s=24, color=color, alpha=0.25, edgecolors="none",
    )
    ax.vlines(
        steps, stats["ci_low"], stats["ci_high"],
        color=color, linewidth=3,
    )
    ax.scatter(
        steps, stats["mean_gap"],
        color=color, s=85, edgecolors="white", zorder=4,
    )

ax.axhline(0, color="#B0B8C3", linewidth=0.8)
ax.set_xticks(plot_summary["n_steps"])
ax.set_xlabel("Trading intervals")
ax.set_ylabel("Rectangular − nonrectangular objective")
ax.set_title("The cost of rectangularization", loc="left", pad=18)
ax.grid(axis="y", color="#ECEFF3")
ax.set_axisbelow(True)

ax.text(
    0.02, 0.97,
    "Faint points: seeds\nSolid points: means\nLines: 95% bootstrap intervals",
    transform=ax.transAxes, va="top",
    fontsize=10, color="#657083",
)

fig.suptitle(subtitle, fontsize=10, color="#778193")
save_figure(fig, "01_gap")

In [ ]:
# Figure 2: empirical gap distributions.

fig, ax = plt.subplots(figsize=(9, 5.2))

for index, steps in enumerate(plot_summary["n_steps"].astype(int)):
    gaps = np.sort(
        good.loc[good["n_steps"] == steps, "gap"].to_numpy()
    )
    fractions = np.arange(1, len(gaps) + 1) / len(gaps)

    ax.step(
        gaps, fractions, where="post",
        color=COLORS[index % len(COLORS)],
        linewidth=2.3,
        label=f"{steps} intervals · {len(gaps)} seeds",
    )

ax.set_xlabel("Rectangularity gap")
ax.set_ylabel("Fraction of seeds with gap ≤ x")
ax.set_ylim(0, 1.03)
ax.set_title("How broadly does the effect appear?", loc="left", pad=18)
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", color="#ECEFF3")
ax.set_axisbelow(True)

fig.suptitle(subtitle, fontsize=10, color="#778193")
save_figure(fig, "02_distribution")

In [ ]:
import shutil

# Make sure the latest tables are on disk.
results.to_csv(output_dir / "results.csv", index=False)
summary.to_csv(output_dir / "summary.csv", index=False)

archive_path = Path(
    shutil.make_archive(
        base_name=str(results_root / output_dir.name),
        format="zip",
        root_dir=str(output_dir.parent),
        base_dir=output_dir.name,
    )
)

print("Results folder:", output_dir.resolve())
print("ZIP archive:   ", archive_path.resolve())
